In [0]:
import pandas as pd

In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
# Cálculo Población ocupada
ocupados
PO = ocupados['FEX_C18'].sum()
PO

In [0]:
# Cálculo Alojamiento y servicios de comida
AS = ocupados[(ocupados['RAMA2D_R4']==55) | (ocupados['RAMA2D_R4']==56)]['FEX_C18'].sum()
AS


In [0]:
# Cálculo Distribución %
D = (AS/PO)*100
D


In [0]:
# Construcción
C = ocupados[(ocupados['RAMA2D_R4']==41) | (ocupados['RAMA2D_R4']==42) | (ocupados['RAMA2D_R4']==43)]['FEX_C18'].sum()
C



In [0]:
# Cálculo Distribución %
D = (C/PO)*100
D

In [0]:
cgse[(cgse['DIRECTORIO']==8637743) & (cgse['SECUENCIA_P']==1) & (cgse['ORDEN']==1) ][['DIRECTORIO','SECUENCIA_P','ORDEN','P3271']]

In [0]:
#Primero hay que anexar la p3271 a la tabla de desempleados
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
## Calcular cantidad de mujeres y hombres desempleados en el quindio

# 1.Cantidad de mujeres desempleadas en el quindio
MDQ = desempleados[(desempleados['P3271']==2) & (desempleados['DSI']==1) & (desempleados['DPTO']==63)]['FEX_C18'].sum()
# 2.Cantidad de hombres desempleados en el quindio
HDQ = desempleados[(desempleados['P3271']==1) & (desempleados['DSI']==1) & (desempleados['DPTO']==63)]['FEX_C18'].sum()

print(f"en el quindio hay {MDQ} Mujeres desempleadas")
print(f"en el quindio hay {HDQ} Hombres desempleados")

In [0]:
## Primero hay que añadir la P3271 al modulo de fuerza de trabajo 
ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
                       on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
## Calcular PEA de mujeres y hombres en el quindio 

# 1.PEA de mujeres en el quindio
PEAMQ = ft[(ft['FT']==1) & (ft['DPTO']==63) & (ft['P3271']==2)]['FEX_C18'].sum()
# 2.PEA de hombres en el quindio
PEAHQ = ft[(ft['FT']==1) & (ft['DPTO']==63) & (ft['P3271']==1) ]['FEX_C18'].sum()

print(f"en el quindio hay {PEAMQ} Mujeres en la PEA")
print(f"en el quindio hay {PEAHQ} Hombres en la PEA")
     

In [0]:
#Por ultimo cuando ya tenemos cantidad de desempleados y PEA podemos calcular el porcentaje de desempleo

TDHQ = HDQ/PEAHQ *100
print(f"El porcentaje de desempleo masculino en el quindio es de {TDHQ}%")
TDMQ = MDQ/PEAMQ *100
print(f"El porcentaje de desempleo femenino en el quindio es de {TDMQ}%")
     